# Generate convergence/coverage maps
Run this after the four training matrices exist. Legacy filenames are retained for compatibility.

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from differentiable_lensing import DifferentiableLensing
LR_GRID_SHAPE=64
HR_GRID_SHAPE=128
DEVICE='cpu'
required=['scatter_to_log_128.pt','forward_from_log_128.pt','scatter_from_log_128.pt','sparse_grid_fracs_euclid_backward.pt']
for p in required: assert Path(p).exists(),f'Missing {p}'

In [ ]:
to_log=torch.load('scatter_to_log_128.pt',map_location=DEVICE).coalesce()
forward_from_log=torch.load('forward_from_log_128.pt',map_location=DEVICE).coalesce()
from_log=torch.load('scatter_from_log_128.pt',map_location=DEVICE).coalesce()
backward=torch.load('sparse_grid_fracs_euclid_backward.pt',map_location=DEVICE).coalesce()
assert to_log.shape==forward_from_log.shape==from_log.shape==(HR_GRID_SHAPE**2,HR_GRID_SHAPE**2)
assert backward.shape==(LR_GRID_SHAPE**2,LR_GRID_SHAPE**2)
lens_hr=DifferentiableLensing(DEVICE,alpha=None,target_resolution=1.0,target_shape=HR_GRID_SHAPE)
lens_lr=DifferentiableLensing(DEVICE,alpha=None,target_resolution=1.0,target_shape=LR_GRID_SHAPE)
ones_hr=torch.ones(1,1,HR_GRID_SHAPE,HR_GRID_SHAPE)
ones_lr=torch.ones(1,1,LR_GRID_SHAPE,LR_GRID_SHAPE)
forward_hr=lens_hr.cross_grid_fill(ones_hr,[to_log,forward_from_log,from_log])
forward_lr=F.interpolate(forward_hr,size=(LR_GRID_SHAPE,LR_GRID_SHAPE),mode='area')
backward_lr=lens_lr.cross_grid_fill(ones_lr,[backward])
source_convergence_map=forward_lr[0,0].clamp(0,1)
image_convergence_map=backward_lr[0,0].clamp(0,1)
torch.save(source_convergence_map,'source_convergence_map.pt')
torch.save(image_convergence_map,'image_convergence_map.pt')
print('saved maps',source_convergence_map.shape,image_convergence_map.shape)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,title,m in [(axes[0],'Forward image-plane coverage',source_convergence_map),(axes[1],'Backward source-plane coverage',image_convergence_map)]:
    im=ax.imshow(m,origin='lower'); ax.set_title(title); fig.colorbar(im,ax=ax)
    print(title,float(m.min()),float(m.max()),float(m.mean()),float((m>0).float().mean()))
plt.tight_layout(); plt.show()